# Notebook 09 — $\Pi$-Model Consistency Regularization

**Session:** Image Augmentation  
**Module:** 02 · Calculus and Linear Algebra

## Goal

Implement **$\Pi$-Model** consistency regularization (Laine & Aila, 2017): evaluate the network on both an augmented sample $x$ and a noise-perturbed counterpart $\tilde{x} = x + \epsilon$, and optimize the composite objective $\mathcal{L}_{\Pi} = \mathcal{L}_{\text{CE}}(f_\theta(x), y) + \alpha \, \|f_\theta(x) - f_\theta(\tilde{x})\|_2^2$.

**Learning objectives:**
- Generate perturbed paired views inside the training loop with detached teacher targets.
- Combine supervised cross-entropy and unsupervised MSE consistency loss.
- Evaluate generalization across K-Fold cross-validation.

In [ ]:
from pathlib import Path

import albumentations
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import KFold
from torch.utils.data import DataLoader, Dataset, SubsetRandomSampler
from torchvision.models import resnet18

## 1  $\Pi$-Model Training Loop

In [ ]:
def get_accuracy(model: nn.Module, data_loader: DataLoader, device: torch.device) -> float:
    """Compute classification accuracy (%) over an entire DataLoader."""
    correct = 0
    total = 0

    model.eval()
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return 100.0 * (correct / total) if total > 0 else 0.0


def plot_losses(
    train_acc: list[float],
    valid_acc: list[float],
    train_loss: list[float],
    valid_loss: list[float],
) -> None:
    """Plot training and validation accuracy and loss curves."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

    ax1.plot(train_acc, color="tab:blue", label="Train Accuracy")
    ax1.plot(valid_acc, color="tab:red", label="Validation Accuracy")
    ax1.set(title="Accuracy over Epochs", xlabel="Epoch", ylabel="Accuracy (%)")
    ax1.grid(True, alpha=0.3)
    ax1.legend()

    ax2.plot(train_loss, color="tab:blue", label="Train Loss")
    ax2.plot(valid_loss, color="tab:red", label="Validation Loss")
    ax2.set(title="Loss over Epochs", xlabel="Epoch", ylabel="Loss")
    ax2.grid(True, alpha=0.3)
    ax2.legend()

    fig.tight_layout()
    plt.show()


def validate(
    valid_loader: DataLoader,
    model: nn.Module,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[nn.Module, float]:
    """Run one validation epoch without gradient tracking."""
    model.eval()
    running_loss = 0.0

    with torch.no_grad():
        for images, labels in valid_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()

    epoch_loss = running_loss / max(len(valid_loader), 1)
    return model, epoch_loss


def train_pi_model(
    train_loader: DataLoader,
    model: nn.Module,
    criterion_ce: nn.Module,
    criterion_mse: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    consistency_weight: float = 0.1,
) -> tuple[nn.Module, torch.optim.Optimizer, float]:
    """Run one training epoch combining Cross-Entropy and perturbation consistency MSE."""
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        perturbed_images = images + torch.randn_like(images) * 0.25

        outputs = model(images)
        with torch.no_grad():
            perturbed_outputs = model(perturbed_images)

        loss_ce = criterion_ce(outputs, labels)
        loss_mse = criterion_mse(outputs, perturbed_outputs)
        loss = loss_ce + consistency_weight * loss_mse
        running_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return model, optimizer, running_loss / max(len(train_loader), 1)


def training_loop(
    model: nn.Module,
    criterion_ce: nn.Module,
    criterion_mse: nn.Module,
    optimizer: torch.optim.Optimizer,
    train_loader: DataLoader,
    valid_loader: DataLoader,
    epochs: int,
    device: torch.device,
    fold_idx: int,
    checkpoint_dir: Path = Path("outputs/pi_model_checkpoints"),
) -> tuple[nn.Module, torch.optim.Optimizer, tuple[list[float], list[float]]]:
    """Train the Pi-Model across epochs and plot accuracy/loss curves."""
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    train_losses: list[float] = []
    valid_losses: list[float] = []
    list_train_acc: list[float] = []
    list_val_acc: list[float] = []

    for epoch in range(epochs):
        model, optimizer, train_loss = train_pi_model(
            train_loader, model, criterion_ce, criterion_mse, optimizer, device
        )
        model, valid_loss = validate(valid_loader, model, criterion_ce, device)
        train_acc = get_accuracy(model, train_loader, device=device)
        valid_acc = get_accuracy(model, valid_loader, device=device)

        print(
            f"Epoch {epoch:02d} | Train loss: {train_loss:.4f} | Valid loss: {valid_loss:.4f} "
            f"| Train acc: {train_acc:.2f}% | Valid acc: {valid_acc:.2f}%"
        )
        list_train_acc.append(train_acc)
        list_val_acc.append(valid_acc)
        train_losses.append(train_loss)
        valid_losses.append(valid_loss)

        torch.save(
            model.state_dict(),
            checkpoint_dir / f"pi_model_fold_{fold_idx + 1}_epoch_{epoch}.pth",
        )

    plot_losses(list_train_acc, list_val_acc, train_losses, valid_losses)
    return model, optimizer, (train_losses, valid_losses)

## 2  Dataset, Model, and K-Fold Execution

In [ ]:
class DogCatDataset(Dataset):
    """Portable image classification dataset reading train/test class subfolders."""

    def __init__(self, root_dir: str | Path, transform=None) -> None:
        self.root_dir = Path(root_dir)
        self.list_images_path: list[Path] = []
        self.list_labels: list[str] = []
        self.class_to_idx = {"dogs": 0, "cats": 1}
        self.transform = transform

        for sub_dir in sorted(self.root_dir.iterdir()):
            if not sub_dir.is_dir() or sub_dir.name not in self.class_to_idx:
                continue
            for image_path in sorted(sub_dir.iterdir()):
                if image_path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                    self.list_images_path.append(image_path)
                    self.list_labels.append(sub_dir.name)

    def __len__(self) -> int:
        return len(self.list_images_path)

    def __getitem__(self, idx: int) -> tuple[np.ndarray, np.ndarray]:
        image = cv2.imread(str(self.list_images_path[idx]))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (224, 224))
        label = np.array(self.class_to_idx[self.list_labels[idx]], dtype=np.int64)

        if self.transform is not None:
            res = self.transform(image=image)
            image = res["image"].astype(np.float32)
        else:
            image = image.astype(np.float32) / 255.0

        image = image.transpose(2, 0, 1)
        return image, label

In [ ]:
class Net(nn.Module):
    """Lightweight ResNet-18 classifier with a 3x3 input stem for 2-class tasks."""

    def __init__(self, num_classes: int = 2) -> None:
        super().__init__()
        self.model = resnet18(weights=None, num_classes=num_classes)
        self.model.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)


def get_transforms() -> tuple[albumentations.Compose, albumentations.Compose]:
    """Create stochastic Albumentations pipelines for consistency training."""
    transforms_train = albumentations.Compose(
        [
            albumentations.HorizontalFlip(p=0.5),
            albumentations.ImageCompression(quality_range=(99, 100), p=0.5),
            albumentations.Affine(
                scale=(0.8, 1.2),
                translate_percent=(-0.2, 0.2),
                rotate=(-10, 10),
                p=0.7,
            ),
            albumentations.Normalize(),
        ]
    )
    transforms_val = albumentations.Compose([albumentations.Normalize()])
    return transforms_train, transforms_val


DATA_DIR = Path("../data/sample_dataset")
transforms_train, transforms_val = get_transforms()
transformed_train_data = DogCatDataset(DATA_DIR / "train", transform=transforms_train)
transformed_test_data = DogCatDataset(DATA_DIR / "test", transform=transforms_val)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
splits = KFold(n_splits=2, shuffle=True, random_state=42)

for fold_idx, (train_idx, val_idx) in enumerate(
    splits.split(np.arange(len(transformed_train_data)))
):
    print(f"\n=== Fold {fold_idx + 1} / 2 ===")
    train_loader = DataLoader(
        transformed_train_data, batch_size=8, sampler=SubsetRandomSampler(train_idx.tolist())
    )
    val_loader = DataLoader(
        transformed_train_data, batch_size=8, sampler=SubsetRandomSampler(val_idx.tolist())
    )
    model = Net().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    loss_ce = nn.CrossEntropyLoss()
    loss_mse = nn.MSELoss()
    model, optimizer, _ = training_loop(
        model,
        loss_ce,
        loss_mse,
        optimizer,
        train_loader,
        val_loader,
        epochs=3,
        device=device,
        fold_idx=fold_idx,
    )